<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# Procesamiento de Lenguaje Natural
## Desafío 4 – Bot QA con LSTM (Encoder–Decoder)


### Objetivo
Utilizar datos del challenge **ConvAI2** para construir un **bot de preguntas y respuestas** (QA) basado en un modelo **encoder–decoder con LSTM**, siguiendo el esquema visto en clase.

Dataset: http://convai.io/data/


In [ ]:
!pip install --quiet gdown

In [ ]:
import os
import re
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

### 1 – Descarga y carga del dataset

In [ ]:
import gdown

if not os.path.exists('data_volunteers.json'):
    url = 'https://drive.google.com/uc?id=1awUxYwImF84MIT5-jCaYAPe2QwSgS1hN'
    gdown.download(url, 'data_volunteers.json', quiet=False)
else:
    print('Dataset ya descargado')

In [ ]:
with open('data_volunteers.json') as f:
    data = json.load(f)

data[0].keys()

### 2 – Preprocesamiento
Se generan pares **pregunta → respuesta**, agregando tokens `<sos>` y `<eos>`.

In [ ]:
def clean_text(txt):
    txt = txt.lower()
    txt = re.sub(r"n't", ' not', txt)
    txt = re.sub(r"'re", ' are', txt)
    txt = re.sub(r"'s", ' is', txt)
    txt = re.sub(r"'m", ' am', txt)
    txt = re.sub(r"\W+", ' ', txt)
    return txt.strip()

input_sentences = []
output_sentences = []
output_sentences_inputs = []

MAX_LEN = 30

for conv in data:
    dialog = conv['dialog']
    for i in range(len(dialog) - 1):
        q = clean_text(dialog[i]['text'])
        a = clean_text(dialog[i + 1]['text'])
        if len(q) <= MAX_LEN and len(a) <= MAX_LEN:
            input_sentences.append(q)
            output_sentences.append(a + ' <eos>')
            output_sentences_inputs.append('<sos> ' + a)

print('Cantidad de pares QA:', len(input_sentences))

### 3 – Tokenización

In [ ]:
MAX_VOCAB = 8000

tokenizer_in = Tokenizer(num_words=MAX_VOCAB)
tokenizer_in.fit_on_texts(input_sentences)
seq_in = tokenizer_in.texts_to_sequences(input_sentences)

tokenizer_out = Tokenizer(num_words=MAX_VOCAB, filters='')
tokenizer_out.fit_on_texts(['<sos>', '<eos>'] + output_sentences)
seq_out = tokenizer_out.texts_to_sequences(output_sentences)
seq_out_in = tokenizer_out.texts_to_sequences(output_sentences_inputs)

max_len_in = max(len(s) for s in seq_in)
max_len_out = max(len(s) for s in seq_out)

encoder_input = pad_sequences(seq_in, maxlen=max_len_in)
decoder_input = pad_sequences(seq_out_in, maxlen=max_len_out, padding='post')
decoder_output = pad_sequences(seq_out, maxlen=max_len_out, padding='post')

num_words_out = min(MAX_VOCAB, len(tokenizer_out.word_index) + 1)
decoder_target = to_categorical(decoder_output, num_classes=num_words_out)

### 4 – Modelo Encoder–Decoder con LSTM

In [ ]:
N_UNITS = 128
EMB_DIM = 128

encoder_inputs = Input(shape=(max_len_in,))
enc_emb = Embedding(input_dim=MAX_VOCAB, output_dim=EMB_DIM)(encoder_inputs)
encoder_lstm = LSTM(N_UNITS, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(max_len_out,))
dec_emb = Embedding(input_dim=num_words_out, output_dim=EMB_DIM)(decoder_inputs)
decoder_lstm = LSTM(N_UNITS, return_sequences=True, return_state=True)
dec_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(num_words_out, activation='softmax')
decoder_outputs = decoder_dense(dec_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

### 5 – Modelos de inferencia

In [ ]:
encoder_model = Model(encoder_inputs, encoder_states)

dec_state_h = Input(shape=(N_UNITS,))
dec_state_c = Input(shape=(N_UNITS,))
dec_states_inputs = [dec_state_h, dec_state_c]

dec_input_single = Input(shape=(1,))
dec_emb_single = dec_emb(dec_input_single)
dec_out, h, c = decoder_lstm(dec_emb_single, initial_state=dec_states_inputs)
dec_states = [h, c]
dec_out = decoder_dense(dec_out)

decoder_model = Model([dec_input_single] + dec_states_inputs, [dec_out] + dec_states)

In [ ]:
idx2word = {v: k for k, v in tokenizer_out.word_index.items()}

def chat(question):
    seq = tokenizer_in.texts_to_sequences([clean_text(question)])
    seq = pad_sequences(seq, maxlen=max_len_in)
    states = encoder_model.predict(seq)
    target = np.array([[tokenizer_out.word_index['<sos>']]])
    result = []
    for _ in range(max_len_out):
        out, h, c = decoder_model.predict([target] + states)
        idx = np.argmax(out[0, 0])
        if idx2word.get(idx) == '<eos>':
            break
        result.append(idx2word.get(idx, ''))
        target = np.array([[idx]])
        states = [h, c]
    return ' '.join(result)

In [ ]:
print(chat('how are you today'))

### Conclusión
Se implementó un bot QA basado en LSTM encoder–decoder siguiendo el esquema visto en clase. El modelo aprende patrones de respuesta simples y constituye una solución base válida para el desafío.